# Ablation sampling diagnostics

**Role:** Analysis.

Compare uniform and identity-stratified recipient sampling on saved validation graphs at a chosen model depth. Use the exact same sampler as the total and size-dependent analyses and inspect every fold, without evaluating model predictions. Enumerate eligible recipient/source-identity opportunities at hops 1 through depth, then report sampled fractions, organoid coverage, variation across sampling seeds and source-cell reuse. Stratified quotas are per recipient identity and may select more cells; comparisons report the actual support rather than implying equal sampling budgets.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Analysis settings

Choose the same training run, depth and recipient quota used for ablation. Sampling repeats assess sampling variation, not new organoids.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Intervention and saved model selection
TRAINING_SUBFOLDER = 'gin_depth_training'  # Workflow folder under training_results/; masking uses fate_masking_training.
TRAINING_RUN = None  # Saved run directory name; None requires one available run.
MODEL_DEPTH = 2  # Analyze exact hops 1..depth; depth 0 has no neighbor interactions.
MODEL_NAME = None  # None requires one family; otherwise gin, film, or gin_film_size for masking runs.
HIDDEN_DIM = None  # None requires one saved width at the chosen depth.
MODEL_SEEDS = None  # None includes every saved seed. Every saved fold is always included.
MASKING_RATE = None  # Select a saved rate for ANY analysis of a masking-trained checkpoint, e.g. .05.

# Coverage sampling (restored from 9ae3657: ablation_analysis)
SAMPLING_SCHEME = 'coverage'  # coverage: pair/hop minimum targets; alternatives: uniform or stratified.
CENTER_SAMPLE_SIZE = 2000  # Maximum recipient centers per validation fold; None keeps all (coverage only).
CENTER_MIN_MARKER_COUNT = 50  # Target centers per recipient identity before non-overlap filtering.
CENTER_MIN_PAIR_COUNT = 25  # Target cases per recipient/source identity at EACH hop before filtering.
SAMPLING_SEED = 0  # Original SUBGRAPH_SEED; fold index is added, identical across model seeds/methods.

# Source-cell independence
CENTER_APPLY_NON_OVERLAP = True  # At hop d, retained sources block others within graph distance d.
CENTER_NON_OVERLAP_GROUP_COLUMNS = ('hop', 'center_marker', 'source_marker')  # Apply independently within each pair/hop.

# Alternative sampling schemes
CENTERS_PER_ORGANOID = 5  # Only uniform/stratified: recipients per organoid (per identity for stratified).

# Coverage experiment
SAMPLING_SCHEMES = [SAMPLING_SCHEME,'uniform','stratified']  # Recipient allocation strategies to compare.
SAMPLING_REPEATS = [SAMPLING_SEED,SAMPLING_SEED+1,SAMPLING_SEED+2]  # Independent sampling draws, not extra biological replicates.
HEATMAP_DISPLAY_MIN_CASES = 20  # Same retained-case threshold as the analysis notebooks; int or per-hop dict.
DEVICE = 'cpu'  # Model restoration device; no prediction is performed.
OUTPUT_TAG = 'default'  # Folder under the saved run's sampling diagnostics.


## Restore held-out graphs

Use one checkpoint per fold to restore validation inputs; other initialization seeds have the same membership.

In [ ]:
from src.artifacts.paths import resolve_training_run
from src.artifacts.runs import AnalysisRun, select_depth_records
from src.analysis.interventions.fate_edits import fate_graphs, sample_fate_contexts, expand_recipients
RUN_DIR = resolve_training_run(ROOT,TRAINING_SUBFOLDER,TRAINING_RUN)
run = AnalysisRun(RUN_DIR,root=ROOT)
records = select_depth_records(run,MODEL_DEPTH,model_name=MODEL_NAME,hidden_dim=HIDDEN_DIM,
    seeds=MODEL_SEEDS,rate=MASKING_RATE)
display(records)
OUTPUT_DIR = RUN_DIR/'analysis'/'sampling_diagnostics'/f'depth{MODEL_DEPTH}'/OUTPUT_TAG
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)


## Census and sampled coverage

Enumerate all eligible recipient/source-identity/hop combinations, including unassigned cells. Then apply each sampling scheme. The eligible-source count records how many physical source cells each opportunity could choose from.

Use the same coverage targets and case-level non-overlap settings as total/size analysis. Targets apply per fold before non-overlap; diagnostics report shortfalls after that filter. The selected schemes have different budgets (global for coverage, per-organoid for alternatives), so this compares the configured designs, not necessarily equal sample sizes. No curvature or anatomical label enters sampling.


In [ ]:
support_rows,case_frames,stage_frames = [],[],[]
# Validation inputs do not depend on initialization seed; inspect each fold once.
for fold,grid in records.groupby('fold'):
    pack = run.select(grid.iloc[0].key,device=DEVICE)
    graphs = fate_graphs(pack)
    # Census of eligible recipient × source-identity × hop opportunities.
    _,population = sample_fate_contexts(graphs,pack['marker_names'],MODEL_DEPTH,
        scheme='uniform',centers=max(len(g.x) for g in graphs),seed=0)
    census = expand_recipients(population)
    keys = ['hop','center_marker','source_marker_name']
    population_counts = census.groupby(keys).agg(eligible_recipients=('case_id','size'),
        eligible_organoids=('organoid_str','nunique'),eligible_source_pairs=('eligible_sources','sum'))
    for scheme in SAMPLING_SCHEMES:
        for seed in SAMPLING_REPEATS:
            _,sample,sampling_info = sample_fate_contexts(graphs,pack['marker_names'],MODEL_DEPTH,
                scheme=scheme,centers=CENTERS_PER_ORGANOID,seed=seed+int(fold),
                max_subgraphs=CENTER_SAMPLE_SIZE,min_center_count=CENTER_MIN_MARKER_COUNT,
                min_pair_count=CENTER_MIN_PAIR_COUNT,apply_non_overlap=CENTER_APPLY_NON_OVERLAP,
                non_overlap_group_columns=CENTER_NON_OVERLAP_GROUP_COLUMNS,return_info=True)
            stages = sampling_info['pair_coverage'].assign(fold=fold,scheme=scheme,sampling_seed=seed)
            threshold = (stages.hop.map(lambda h: HEATMAP_DISPLAY_MIN_CASES.get(h,HEATMAP_DISPLAY_MIN_CASES.get(str(h),0)))
                if isinstance(HEATMAP_DISPLAY_MIN_CASES,dict) else HEATMAP_DISPLAY_MIN_CASES)
            stages['passes_display_threshold'] = stages.selected_count >= threshold
            stage_frames.append(stages)
            if sample.empty:
                continue
            expanded = expand_recipients(sample)
            sampled = expanded.groupby(keys).agg(sampled_recipients=('case_id','size'),sampled_organoids=('organoid_str','nunique'))
            report = population_counts.join(sampled).fillna(0).reset_index()
            report['recipient_coverage'] = report.sampled_recipients/report.eligible_recipients
            report['organoid_coverage'] = report.sampled_organoids/report.eligible_organoids
            report['fold'],report['scheme'],report['sampling_seed'] = fold,scheme,seed
            support_rows.append(report)
            sample['fold'],sample['scheme'],sample['sampling_seed'] = fold,scheme,seed
            case_frames.append(sample)
coverage = pd.concat(support_rows,ignore_index=True)
sampled_cases = pd.concat(case_frames,ignore_index=True)
coverage.to_csv(OUTPUT_DIR/'coverage.csv',index=False)
sampled_cases.to_csv(OUTPUT_DIR/'sampled_cases.csv.gz',index=False)
display(coverage)

sampling_stages = pd.concat(stage_frames,ignore_index=True)
sampling_stages.to_csv(OUTPUT_DIR/'sampling_stages.csv',index=False)
display(sampling_stages)
(OUTPUT_DIR/'settings.json').write_text(json.dumps(dict(
    training_run=str(RUN_DIR),model_depth=MODEL_DEPTH,model_keys=records.key.tolist(),
    sampling_schemes=SAMPLING_SCHEMES,sampling_repeats=SAMPLING_REPEATS,
    center_sample_size=CENTER_SAMPLE_SIZE,center_min_marker_count=CENTER_MIN_MARKER_COUNT,
    center_min_pair_count=CENTER_MIN_PAIR_COUNT,centers=CENTERS_PER_ORGANOID,
    center_apply_non_overlap=CENTER_APPLY_NON_OVERLAP,
    center_non_overlap_group_columns=list(CENTER_NON_OVERLAP_GROUP_COLUMNS),
    heatmap_display_min_cases=HEATMAP_DISPLAY_MIN_CASES),indent=2))


## Coverage and source reuse

Plot sampled fractions by distance and reuse of physical sources. The band is the range over folds and sampling draws, not a biological confidence interval.

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(12,4))
for scheme,frame in coverage.groupby('scheme'):
    curve = frame.groupby('hop').recipient_coverage.agg(['mean','min','max'])
    line, = axes[0].plot(curve.index,curve['mean'],'o-',label=scheme)
    axes[0].fill_between(curve.index,curve['min'],curve['max'],color=line.get_color(),alpha=.15)
for scheme,frame in sampled_cases.groupby('scheme'):
    reuse = frame.groupby(['fold','sampling_seed','organoid_str','orig_source_node']).orig_center.nunique()
    axes[1].hist(reuse,bins=20,alpha=.4,label=scheme)
axes[0].set(xlabel='Exact hop',ylabel='Sampled / eligible recipient opportunities',title='Mean and range across folds/draws')
axes[1].set(xlabel='Recipient uses per physical source cell',ylabel='Source cells',title='Source reuse')
for ax in axes:ax.legend()
plt.tight_layout();plt.show()
